In [ ]:
# ======================================================================
# PROJECT 2 - Teaching a neural network to read handwriting
# AI Builders Lab  ·  Class 2  ·  23 August
#
# CODE ONLY. Every cell, no explanations.
# The explained version, with diagrams, is:  02_Handwriting_MNIST_ANN.ipynb
#
# Run cells with Shift + Enter, in order, top to bottom.
# The last cell draws a box you can write in with the mouse.
# ======================================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)
print("Toolbox is open.")

In [ ]:
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# X = the images (the questions).   Y = the correct digit (the answers).
print("Training images:", X_train.shape)   # (60000, 28, 28) -> 60000 images, each 28x28 pixels
print("Training labels:", Y_train.shape)   # (60000,)
print("Testing images: ", X_test.shape)    # (10000, 28, 28) -> the sealed exam
print("Testing labels: ", Y_test.shape)

In [ ]:
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[i], cmap='gray')   # cmap='gray' - these are greyscale, not colour
    plt.title(Y_train[i])                 # the correct answer, printed on top
    plt.axis('off')
plt.suptitle("The first 10 training images, with their labels")
plt.show()

In [ ]:
# Print image 0 as text: '#' for bright ink, '+' for faint, '.' for empty paper.
print("This image is labelled:", Y_train[0], "\n")
for row in X_train[0]:
    print("".join("#" if p > 128 else ("+" if p > 40 else ".") for p in row))

In [ ]:
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print("Smallest pixel value now:", X_train.min())
print("Largest pixel value now: ", X_train.max())

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28)),              # tell the model what one image looks like
    layers.Flatten(),                          # 28x28 square -> 784 numbers in a row
    layers.Dense(128, activation='relu'),      # hidden layer 1
    layers.Dropout(0.2),                       # randomly ignore 20% of neurons while training
    layers.Dense(64,  activation='relu'),      # hidden layer 2
    layers.Dense(10,  activation='softmax')    # 10 outputs = 10 digits, as probabilities
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled. The rules of learning are set.")

In [ ]:
history = model.fit(
    X_train, Y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_test, Y_test),
    verbose=1
)

print("\nTraining finished.")

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)

print("ACCURACY on unseen data:", round(test_accuracy * 100, 2), "%")
print("\nOut of 10,000 digits it had never seen, it got about",
      int(test_accuracy * 10000), "right.")

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='training')
plt.plot(history.history['val_accuracy'], label='unseen test data')
plt.title('Accuracy (higher is better)'); plt.xlabel('Epoch'); plt.legend(loc='lower right')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='unseen test data')
plt.title('Loss (lower is better)'); plt.xlabel('Epoch'); plt.legend(loc='upper right')
plt.grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
predictions = model.predict(X_test, verbose=0)

i = 9        # CHANGE THIS NUMBER and re-run to inspect a different test image

probabilities = predictions[i]
guess = int(np.argmax(probabilities))     # argmax = "which position holds the biggest number?"

plt.figure(figsize=(9, 3.2))
plt.subplot(1, 2, 1)
plt.imshow(X_test[i], cmap='gray'); plt.axis('off')
plt.title(f"true answer: {Y_test[i]}   model says: {guess}")
plt.subplot(1, 2, 2)
plt.bar(range(10), probabilities)
plt.xticks(range(10)); plt.ylim(0, 1)
plt.title("how sure it is about each digit")
plt.tight_layout(); plt.show()

print("Confidence in its answer:", round(float(probabilities[guess]) * 100, 2), "%")

In [ ]:
model.save('handwriting_model.keras')
print("Saved to handwriting_model.keras")

# Click the FOLDER icon in Colab's left sidebar to see the file.
# Colab wipes this storage when the session ends - right-click -> Download to keep it.

In [ ]:
from PIL import Image

def prepare_digit(img, show=True):
    """Convert ANY picture of a single digit into the 28x28 format the model expects."""

    original = img
    g = np.array(img.convert("L"), dtype=np.float32)   # "L" = convert to greyscale

    # STEP 1: Flip black-on-white to white-on-black.
    # MNIST is white ink on black paper. Your notebook paper is the opposite.
    # We check the brightness of the border pixels: if the edge of the picture is
    # light, we are looking at paper, so we invert.
    border = np.concatenate([g[0, :], g[-1, :], g[:, 0], g[:, -1]])
    if border.mean() > 127:
        g = 255.0 - g

    # STEP 2: Stretch the contrast and delete the faint stuff.
    # Photos have shadows, paper texture, and grey smudges. Anything dimmer than
    # 40% of the brightest ink is treated as background and set to pure black.
    g = g - g.min()
    if g.max() > 0:
        g = g / g.max() * 255.0
    g[g < 0.4 * g.max()] = 0

    # STEP 3: Crop away the empty space, keeping only the ink.
    ys, xs = np.nonzero(g)
    if len(ys) == 0:
        print("I can't find any ink in this image. Try drawing darker or thicker.")
        return np.zeros((28, 28), dtype=np.float32)
    g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    # STEP 4: Resize so the longest side is 20 pixels.
    # Why 20 and not 28? Because the people who built MNIST scaled every digit into
    # a 20x20 box and left a 4-pixel margin. We copy them exactly. Get this wrong
    # and your digit is the wrong size compared to everything the model studied.
    h, w = g.shape
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    g = np.array(Image.fromarray(g.astype(np.uint8)).resize((new_w, new_h), Image.LANCZOS),
                 dtype=np.float32)

    # STEP 5: Paste into a 28x28 black square, centred by CENTRE OF MASS.
    # MNIST centres each digit on its centre of gravity, not its bounding box.
    # This is the step everyone skips, and skipping it is the #1 reason
    # "it works on MNIST but not on my own writing".
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = g
    cy, cx = np.array(np.nonzero(canvas)).mean(axis=1)
    canvas = np.roll(canvas, int(round(13.5 - cy)), axis=0)
    canvas = np.roll(canvas, int(round(13.5 - cx)), axis=1)

    # STEP 6: Scale to 0-1, exactly as we did to the training data.
    canvas = canvas / 255.0

    if show:
        plt.figure(figsize=(7, 3))
        plt.subplot(1, 2, 1); plt.imshow(original, cmap='gray')
        plt.title("What you gave it"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(canvas, cmap='gray')
        plt.title("What the model actually sees"); plt.axis('off')
        plt.show()

    return canvas


def predict_digit(img28):
    """Feed a prepared 28x28 image to the model and show the verdict."""
    p = model.predict(img28.reshape(1, 28, 28), verbose=0)[0]
    guess = int(np.argmax(p))

    print("=" * 44)
    print("   THE MODEL SAYS:", guess, "  (", round(p[guess] * 100, 1), "% confident )")
    print("=" * 44)
    print("\nIts full opinion:")
    for digit in range(10):
        bar = "#" * int(p[digit] * 40)
        print(f"  {digit} | {bar:<40} {p[digit]*100:5.1f}%")
    return guess

print("Helper functions ready.")

In [ ]:
# ONE cell: draw, click DONE, and the model answers. Nothing leaves this page.

from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io

canvas_html = '''
<div style="font-family: sans-serif; color:#10162F;">
  <canvas id="pad" width="300" height="300"
          style="border:3px solid #555; background:#000; cursor:crosshair; touch-action:none;"></canvas>
  <div style="margin-top:10px;">
    <button id="done"  style="font-size:16px; padding:9px 20px; cursor:pointer;">DONE &mdash; read my digit</button>
    <button id="clear" style="font-size:16px; padding:9px 20px; cursor:pointer;">Clear</button>
  </div>
  <p style="font-size:13px; color:#666; margin-top:8px;">
     Draw <b>one</b> digit, <b>big</b> and <b>thick</b>, filling most of the box.
     Mouse, trackpad, finger or stylus all work.
  </p>
</div>
<script>
  var c   = document.getElementById('pad');
  var ctx = c.getContext('2d');
  ctx.fillStyle = 'black';
  ctx.fillRect(0, 0, c.width, c.height);
  ctx.strokeStyle = 'white';
  ctx.lineWidth   = 22;          // thick, so the stroke survives shrinking to 28x28
  ctx.lineCap     = 'round';
  ctx.lineJoin    = 'round';

  var drawing = false;
  function spot(e) {
    var r = c.getBoundingClientRect();
    return [(e.clientX - r.left) * c.width / r.width,
            (e.clientY - r.top)  * c.height / r.height];
  }
  // pointer events cover mouse, trackpad, touchscreen and stylus in one go
  c.addEventListener('pointerdown', function(e) {
    e.preventDefault();
    drawing = true;
    var p = spot(e);
    ctx.beginPath(); ctx.moveTo(p[0], p[1]); ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  c.addEventListener('pointermove', function(e) {
    if (!drawing) return;
    e.preventDefault();
    var p = spot(e);
    ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  window.addEventListener('pointerup', function() { drawing = false; });

  document.getElementById('clear').onclick = function() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, c.width, c.height);
  };

  // Python waits on this promise until you click DONE
  var data = new Promise(function(resolve) {
    document.getElementById('done').onclick = function() {
      resolve(c.toDataURL('image/png'));
    };
  });
</script>
'''

display(HTML(canvas_html))

# Python pauses here until you click DONE. The timeout means it can never hang forever:
# if nobody clicks, it gives up after 10 minutes instead of freezing the notebook.
def wait_for_drawing():
    try:
        return eval_js("data", timeout_sec=600)
    except TypeError:
        return eval_js("data")      # older Colab builds have no timeout option

try:
    drawing_data = wait_for_drawing()
except Exception as error:
    drawing_data = None
    print("Nothing came back from the drawing pad.")
    print("You probably did not click DONE, or the cell was stopped.")
    print("Just run this cell again and draw another digit.")
    print("  (technical detail:", type(error).__name__, ")")

if drawing_data:
    my_drawing = Image.open(io.BytesIO(b64decode(drawing_data.split(',')[1])))

    if np.array(my_drawing.convert("L")).max() < 10:
        print("The pad was empty. Run this cell again, draw a digit, THEN click DONE.")
    else:
        prepared = prepare_digit(my_drawing)   # squeeze it into MNIST's exact format
        predict_digit(prepared)                # and ask the model